In [82]:
import pandas as pd


In [83]:
path="C:\\Users\\TWB\\Desktop\\openClassrooms\\Projet_10\\data\\"

erp = pd.read_excel(path + "Fichier_erp.xlsx", sheet_name="Sheet1")
web = pd.read_excel(path + "Fichier_web.xlsx", sheet_name="Sheet1")
liaison = pd.read_excel(path + "Fichier_liaison.xlsx", sheet_name="Sheet1")

C:\Users\TWB\Desktop\openClassrooms\Projet_9\Projet_10\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
C:\Users\TWB\Desktop\openClassrooms\Projet_9\Projet_10\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)
C:\Users\TWB\Desktop\openClassrooms\Projet_9\Projet_10\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Unknown extension is not supported and will be removed
  warn(msg)


## Script Python d’identification des vins millésimes et vins ordinaires avec le z-score. Pour rappel : 
## z-score = (prix du vin - moyenne prix des vins)/(écart type des prix des vins) 
## un vin millésime est défini comme ayant un z-score  > 2

In [84]:
# Calcul du z-score de la colonne "price" et création de la colonne "price_z-score"
moyenne = erp["price"].mean()
ecartType = erp["price"].std()
erp["price_z-score"] = (erp["price"] - moyenne) / ecartType

# Création de la colonne "millesime" en fonction de la colonne "price_z-score"
erp["millesime"] = False
erp.loc[erp["price_z-score"] > 2, "millesime"] = True

# Code polars
#erp = erp.with_columns(((pl.col("price") - moyenne) / ecartType).alias("z-score"))
#erp = erp.with_columns([pl.lit(True).alias("millesime")])
#erp = erp.with_columns(pl.when(pl.col("z-score") > 2).then(pl.col("millesime") == True))

erp.loc[erp["price_z-score"] > 2, :].to_excel(path + "vinsMillesimes.xlsx", sheet_name="Sheet1")

     product_id  onsale_web  price  stock_quantity stock_status  \
19         4055           0   86.1               1   outofstock   
66         4115           1  100.0              11      instock   
68         4132           1   88.4               5      instock   
208        4352           1  225.0               0   outofstock   
210        4355           1  126.5               2      instock   

     price_z-score  millesime  
19        2.001918       True  
66        2.519951       True  
68        2.087635       True  
208       7.178520       True  
210       3.507567       True  


In [85]:
# Calcul du z-score de la colonne "price" et création de la colonne "price_z-score"
moyenne = erp["price"].mean()
ecartType = erp["price"].std()
erp["price_z-score"] = (erp["price"] - moyenne) / ecartType

# Création de la colonne "millesime" en fonction de la colonne "price_z-score"
erp["millesime"] = False
erp.loc[erp["price_z-score"] > 2, "millesime"] = True

erp.loc[erp["price_z-score"] <= 2, :].to_excel(path + "vinsStandards.xlsx", sheet_name="Sheet1")

## Script d'initiation des tables

In [283]:
import duckdb
conn = duckdb.connect()

conn.sql("ATTACH 'openclassrooms.db'")

conn.sql("DROP TABLE IF EXISTS openclassrooms.erp")
conn.sql("CREATE TABLE openclassrooms.erp AS SELECT * FROM read_xlsx('C:\\Users\\TWB\\Desktop\\openClassrooms\\Projet_10\\data\\Flow_01\\Fichier_erp.xlsx', all_varchar = true, sheet = 'Sheet1')")

conn.sql("DROP TABLE IF EXISTS openclassrooms.web")
conn.sql("CREATE TABLE openclassrooms.web AS SELECT * FROM read_xlsx('C:\\Users\\TWB\\Desktop\\openClassrooms\\Projet_10\\data\\Flow_01\\Fichier_web.xlsx', all_varchar = true, sheet = 'Sheet1')")

conn.sql("DROP TABLE IF EXISTS openclassrooms.liaison")
conn.sql("CREATE TABLE openclassrooms.liaison AS SELECT * FROM read_xlsx('C:\\Users\\TWB\\Desktop\\openClassrooms\\Projet_10\\data\\Flow_01\\Fichier_liaison.xlsx', all_varchar = true, sheet = 'Sheet1')")

print(conn.sql("select * from read_xlsx('C:\\Users\\TWB\\Desktop\\openClassrooms\\Projet_10\\data\\Flow_01\\Fichier_erp.xlsx', all_varchar = true, sheet = 'Sheet1') where product_id = 5483"))

conn.sql("DETACH openclassrooms")

┌────────────┬────────────┬────────────────────┬────────────────┬──────────────┐
│ product_id │ onsale_web │       price        │ stock_quantity │ stock_status │
│  varchar   │  varchar   │      varchar       │    varchar     │   varchar    │
├────────────┼────────────┼────────────────────┼────────────────┼──────────────┤
│ 5483       │ 1          │ 17.899999999999999 │ 22             │ instock      │
└────────────┴────────────┴────────────────────┴────────────────┴──────────────┘



## Script SQL de dédoublonnage de Fichier_web.xlsx avec DuckDB

In [293]:
#conn.sql("ATTACH 'openclassrooms.db'")

#conn.sql("ALTER TABLE openclassrooms.erp ADD COLUMN id UUID;")
#conn.sql("UPDATE openclassrooms.erp set id=gen_random_uuid();")
#conn.sql("alter table openclassrooms.erp add primary key (id);")

#conn.sql("ALTER TABLE openclassrooms.web ADD COLUMN id UUID;")
#conn.sql("UPDATE openclassrooms.web set id=gen_random_uuid();")
#conn.sql("alter table openclassrooms.web add primary key (id);")

#conn.sql("SELECT * FROM openclassrooms.web;")

# Création de la table webDedoublonnage
conn.sql("DROP TABLE IF EXISTS openclassrooms.webDedoublonnage")
conn.sql("CREATE TABLE openclassrooms.webDedoublonnage AS SELECT * FROM openclassrooms.web ORDER BY sku")

# Suppression des colonnes sans importance
conn.sql("ALTER TABLE openclassrooms.webDedoublonnage DROP COLUMN tax_status;")
conn.sql("ALTER TABLE openclassrooms.webDedoublonnage DROP COLUMN post_excerpt;")
conn.sql("ALTER TABLE openclassrooms.webDedoublonnage DROP COLUMN guid;")
conn.sql("ALTER TABLE openclassrooms.webDedoublonnage DROP COLUMN post_type;")
conn.sql("ALTER TABLE openclassrooms.webDedoublonnage DROP COLUMN post_mime_type;")
print(conn.sql("SELECT COUNT(*) FROM openclassrooms.webDedoublonnage;"))






# Suppréssion des lignes qui n'ont pas d'id (sku)
conn.sql("DELETE FROM openclassrooms.webDedoublonnage WHERE sku IS NULL;")
print(conn.sql("SELECT COUNT(*) FROM openclassrooms.webDedoublonnage;"))
# Récupérer deux lignes avec des ventes pour le calcul du chiffres d'affaires







# Select des lignes distinctes (suppréssion des doublons)
#conn.sql("DELETE FROM openclassrooms.webDedoublonnage WHERE sku IS NULL")
conn.sql("CREATE OR REPLACE TABLE openclassrooms.webDedoublonnage AS SELECT DISTINCT * FROM openclassrooms.webDedoublonnage")
print(conn.sql("SELECT COUNT(*) FROM openclassrooms.webDedoublonnage;"))

# Ajout d'un identifiant unique
conn.sql("ALTER TABLE openclassrooms.webDedoublonnage ADD COLUMN id UUID;")
conn.sql("UPDATE openclassrooms.webDedoublonnage set id=gen_random_uuid();")

# Récupération des lignes toujours en double (avec des différences dans d'autres colonnes et les arranger manuellement)
print(conn.sql("SELECT sku FROM openclassrooms.webDedoublonnage WHERE id NOT IN (SELECT DISTINCT ON(sku) id FROM openclassrooms.webDedoublonnage)"))
print(conn.sql("SELECT COUNT(*) FROM openclassrooms.webDedoublonnage;"))
# On constate que les lignes : 1662, 16416, 7818 ont une différence sur les quantités des ventes (on récupère la valeur la plus haute)
# 1662 : 7 - 87  ----- 7 * 49
conn.sql("UPDATE openclassrooms.webDedoublonnage set total_sales='87' where sku = '1662';")
# 7818 : 96 - 6  ----- 96 * 49
conn.sql("UPDATE openclassrooms.webDedoublonnage set total_sales='96' where sku = '7818';")
# 16416 : 2 - 62  ----- 62 * 16.60
conn.sql("UPDATE openclassrooms.webDedoublonnage set total_sales='62' where sku = '16416';")


# Suppréssion des doublons selon le sku
conn.sql("CREATE OR REPLACE TABLE openclassrooms.webDedoublonnage AS SELECT DISTINCT ON(sku) * FROM openclassrooms.webDedoublonnage")
print(conn.sql("SELECT COUNT(*) FROM openclassrooms.webDedoublonnage;"))

conn.sql("DETACH openclassrooms")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1513 │
└──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1428 │
└──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          717 │
└──────────────┘

┌─────────┐
│   sku   │
│ varchar │
├─────────┤
│ 1662    │
│ 16416   │
│ 7818    │
└─────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          717 │
└──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          714 │
└──────────────┘



## Script SQL de suppréssion des doublons du Fichier_erp.xlsx avec DuckDB

In [296]:
conn.sql("ATTACH 'openclassrooms.db'")

# Création de la table erpDedoublonnage
conn.sql("DROP TABLE IF EXISTS openclassrooms.erpDedoublonnage")
conn.sql("CREATE TABLE openclassrooms.erpDedoublonnage AS SELECT * FROM openclassrooms.erp ORDER BY product_id")
print(conn.sql("SELECT COUNT(*) FROM openclassrooms.erpDedoublonnage;"))

# Suppréssion des doublons selon product_id
conn.sql("CREATE OR REPLACE TABLE openclassrooms.erpDedoublonnage AS SELECT DISTINCT ON(product_id) * FROM openclassrooms.erpDedoublonnage")
print(conn.sql("SELECT COUNT(*) FROM openclassrooms.erpDedoublonnage;"))
# Aucun doublon détectés
#5483
print(conn.sql("select * from openclassrooms.erpDedoublonnage where product_id = 5483"))
conn.sql("DETACH openclassrooms")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          825 │
└──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          825 │
└──────────────┘

┌────────────┬────────────┬────────────────────┬────────────────┬──────────────┐
│ product_id │ onsale_web │       price        │ stock_quantity │ stock_status │
│  varchar   │  varchar   │      varchar       │    varchar     │   varchar    │
├────────────┼────────────┼────────────────────┼────────────────┼──────────────┤
│ 5483       │ 1          │ 17.899999999999999 │ 22             │ instock      │
└────────────┴────────────┴────────────────────┴────────────────┴──────────────┘



## Suppression des lignes innutiles (fichier_liaison.xlsx)

In [248]:
conn.sql("ATTACH 'openclassrooms.db'")

# Création de la table liaisonNettoye
conn.sql("DROP TABLE IF EXISTS openclassrooms.liaisonNettoye")
conn.sql("CREATE TABLE openclassrooms.liaisonNettoye AS SELECT * FROM openclassrooms.liaison")
print(conn.sql("SELECT COUNT(*) FROM openclassrooms.liaisonNettoye;"))

# Suppréssion des lignes avec une valeur vide
conn.sql("DELETE FROM openclassrooms.liaisonNettoye WHERE product_id IS NULL;")
conn.sql("DELETE FROM openclassrooms.liaisonNettoye WHERE id_web IS NULL;")
print(conn.sql("SELECT COUNT(*) FROM openclassrooms.liaisonNettoye;"))

conn.sql("DETACH openclassrooms")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          825 │
└──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          734 │
└──────────────┘



## Script SQL de fusion des deux systèmes avec DuckDB

In [298]:
conn.sql("ATTACH 'openclassrooms.db'")

# Création de la table tabFinal
conn.sql("DROP TABLE IF EXISTS openclassrooms.tabFinal")
conn.sql("CREATE TABLE openclassrooms.tabFinal AS SELECT * FROM openclassrooms.webDedoublonnage AS tab1 JOIN openclassrooms.liaisonNettoye AS tab2 ON tab1.sku = tab2.id_web")
print(conn.sql("SELECT COUNT(*) FROM openclassrooms.tabFinal;"))

#conn.sql("DELETE FROM openclassrooms.tabFinal WHERE product_id IS NULL;")
#print(conn.sql("SELECT COUNT(*) FROM openclassrooms.tabFinal;"))

#conn.sql("DELETE FROM openclassrooms.tabFinal WHERE id_web IS NULL;")
#print(conn.sql("SELECT COUNT(*) FROM openclassrooms.tabFinal;"))

conn.sql("CREATE OR REPLACE TABLE openclassrooms.tabFinal AS SELECT * FROM openclassrooms.tabFinal AS tab1 JOIN openclassrooms.erpDedoublonnage AS tab2 ON tab1.product_id = tab2.product_id")
print(conn.sql("SELECT COUNT(*) FROM openclassrooms.tabFinal;"))

# Suppression des colonnes innutiles
conn.sql("ALTER TABLE openclassrooms.tabFinal DROP product_id_1;")
conn.sql("ALTER TABLE openclassrooms.tabFinal DROP sku;")


#Aucune ligne n'a été supprimé par rapport au fichier_web, cela semble bon

#conn.sql("DETACH openclassrooms")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          714 │
└──────────────┘

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│          714 │
└──────────────┘



## Création du fichier_final.xlsx

In [270]:
#conn.sql("ATTACH 'openclassrooms.db'")

#conn.sql("INSTALL excel;")
conn.sql("LOAD excel;")

conn.sql("COPY openclassrooms.tabFinal TO '/data/output.xlsx' WITH (FORMAT xlsx, HEADER true);")


## Une extraction du rapport avec les chiffres d’affaires par produit en .xls

In [302]:
#conn.sql("ATTACH 'openclassrooms.db'")

#conn.sql("ALTER TABLE openclassrooms.tabFinal ADD COLUMN chiffreAffaire double;")
conn.sql("UPDATE openclassrooms.tabFinal set chiffreAffaire=CAST(total_sales AS DOUBLE) * CAST(price AS DOUBLE);")

conn.sql("LOAD excel;")

conn.sql("COPY openclassrooms.tabFinal TO '/data/outputChiffreAffaireByBotlle.xlsx' WITH (FORMAT xlsx, HEADER true);")

print(conn.sql("SELECT * FROM openclassrooms.tabFinal;"))

┌─────────┬──────────────┬──────────────┬────────────────┬─────────────┬───────────┬─────────────┬────────────────────┬────────────────────┬──────────────┬───────────────────────────────────────────────────────────────────┬─────────────┬────────────────┬─────────────┬───────────────┬────────────────────────────────────────────────────────────┬────────────────────┬────────────────────┬───────────────────────┬─────────────┬────────────┬───────────────┬──────────────────────────────────────┬────────────┬─────────┬────────────┬────────────────────┬────────────────┬──────────────┬────────────────────┐
│ virtual │ downloadable │ rating_count │ average_rating │ total_sales │ tax_class │ post_author │     post_date      │   post_date_gmt    │ post_content │                            post_title                             │ post_status │ comment_status │ ping_status │ post_password │                         post_name                          │   post_modified    │ post_modified_gmt  │ post_con

## Script SQL de calcul du chiffre d'affaires global

In [304]:
#conn.sql("ATTACH 'openclassrooms.db'")

conn.sql("COPY (SELECT SUM(chiffreAffaire) FROM openclassrooms.tabFinal) TO '/data/chiffreAffaireTotal.xlsx' WITH (FORMAT xlsx, HEADER true);")

#print(conn.sql("SELECT * FROM openclassrooms.tabFinal;"))

# Après exécution du workflow:

## Une extraction de la liste des vins premium en .csv

## Une extraction des vins ordinaires en .csv

## Une planification de l'exécution de l’ensemble du workflow tous les 15 du mois à 9h en utilisant le mécanisme du trigger et du cron (voir Crontab.guru)